# OE4: reproducción autónoma en Google Colab

**Objetivo oficial (literal).** Medir y comparar el desempeño computacional del artefacto tecnológico desarrollado bajo diferentes condiciones de simulación, utilizando como indicadores el tiempo de convergencia, la latencia y el throughput.

**Qué comprueba este cuaderno.** Comprueba el objetivo en dos etapas ejecutadas en orden: etapa 1, recálculo de los indicadores por corrida (216 corridas válidas) desde cada petición registrada y encadenamiento con la tabla por celda; etapa 2, análisis pre-registrado (ANOVA, contrastes pareados con Holm, comprobaciones no paramétricas, estabilidad y sensibilidad) y exploratorio de interferencia desde los indicadores por corrida.

**Datos de entrada.** Registros por petición de las 216 corridas, `runs.jsonl`, `intentos.jsonl`, monitor de recursos de la sesión, corridas de OE2-B como referencia descriptiva y los scripts de análisis, con los resultados locales para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Pulse «Ejecutar todo» en un entorno nuevo de Colab: los datos se descargan solos desde GitHub (commit fijado) y se verifican por SHA-256; no hay que subir ni buscar archivos.

**Contenido.** 1 descripción · 2 entorno y código (clonado de la etiqueta inmutable) · 3 descarga automática y verificación SHA-256 · 4 validación de integridad y estructura · 5 preprocesamiento y cálculo de indicadores (scripts sellados) · 6 análisis estadístico · 7 comparación con el resultado local y con las cifras de la tesis · 8 tablas · 9 gráficos · 10 pruebas automáticas · 11 resumen. Las diferencias se clasifican como exacta, dentro de tolerancia, explicada por versión, no explicada o no reproducible.

## 1. Entorno y código de análisis
El código vive en el repositorio público (módulos legibles en `reproducibility/etapa26_colab_autonomo/src/repro/`); los scripts de análisis sellados vienen dentro de los paquetes de datos, con su hash verificado.

In [ ]:
import os, sys, platform, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Renato5Lara/multiagent-llm-education.git'
REPO_REF = 'repro-etapa26-v1'            # etiqueta inmutable del codigo; los datos estan fijados por commit en manifests/data_manifest.json
if os.environ.get('REPRO_LOCAL_REPO'):   # solo para pruebas locales antes de publicar la etiqueta
    REPO_DIR = os.environ['REPRO_LOCAL_REPO']
else:
    REPO_DIR = 'repo_repro'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', REPO_REF, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'reproducibility', 'etapa26_colab_autonomo', 'src'))
import repro
from repro import sh
print('Codigo de analisis:', REPO_URL, '@', REPO_REF)


In [ ]:
try:
    import statsmodels
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'statsmodels'], check=True)
    import statsmodels
import statsmodels, numpy, scipy, pandas, matplotlib
ENTORNO = dict(python=platform.python_version(), plataforma=platform.platform(), numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__, matplotlib=matplotlib.__version__, cpu=os.cpu_count(), statsmodels=statsmodels.__version__)
print(ENTORNO)


## 2. Descarga automática y verificación de los paquetes (SHA-256)

In [ ]:
# Descarga automatica desde GitHub (commit fijado) y verificacion SHA-256 contra el manifiesto. No hay que subir nada.
# Si falta un archivo, no se puede descargar o su hash no coincide, se detiene con el archivo, la URL y el hash esperado.
PAQUETES = ['oe4_colab_paquete', 'oe4_http_p1', 'oe4_http_p2', 'oe4_http_p3', 'oe4_http_p4']
repro.fetch_packages(PAQUETES, dest='.')


## 3. Verificación interna, cálculo y análisis (celdas del notebook de comprobación publicado, sin cambios salvo la ejecución de scripts)

## Etapa 1: indicadores por corrida desde las peticiones registradas

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('http/hashes_paquete_http_oe4.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); n+=1
    if sha(os.path.join('http',nm))!=h: bad.append(nm)
print('archivos del paquete HTTP verificados contra el hash local:', n, '| distintos:', bad[:5]); assert not bad
base='http/oe4/campana_01'; ok=0; mal=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join(base,nm)
    if os.path.exists(p):
        ok+=1
        if sha(p)!=h: mal.append(nm)
print('oe4: archivos de la campaña presentes y verificados contra su SHA256SUMS.txt:', ok, '| distintos:', mal); assert not mal

In [ ]:
os.makedirs('http/out', exist_ok=True)
for k in ('oe4',):
    sh(f"python http/recomputo_http.py http/{k}/campana_01 http/out/recomputo_{k}.csv http/out/resumen_{k}.json > http/out/log_{k}.txt 2>&1")
import json
print('Recálculo INDEPENDIENTE, desde cada petición registrada por el generador de carga, de los indicadores por corrida (12 campos) y comparación con lo guardado por el ejecutor:')
for k in ('oe4',):
    r=json.load(open(f'http/out/resumen_{k}.json'))
    mx=max(r['diferencia_abs_max_por_campo'].values()); mr=max(r['diferencia_rel_max_por_campo'].values())
    print(f"{k}: intentos recalculados {r['intentos_con_raw_recalculados']} | corridas válidas finales recalculadas {r['corridas_validas_finales_recalculadas']} | con los 12 campos idénticos {r['corridas_validas_finales_con_todos_los_campos_iguales']} | diferencia absoluta máxima {mx} | relativa máxima {mr}")

In [ ]:
import pandas as pd, math
def runs_de(k): return {json.loads(l)['label']: json.loads(l) for l in open(f'http/{k}/campana_01/runs.jsonl', encoding='utf8') if l.strip()}
# OE4: tabla por celda (media geométrica de P95 y de T_conv, media de throughput, peticiones y errores) desde los valores RECALCULADOS
rc=pd.read_csv('http/out/recomputo_oe4.csv'); rc=rc[rc['valida_final']==True].copy(); rr=runs_de('oe4')
for c,k in (('carga','load'),('N','n_particles'),('workers','workers')): rc[c]=rc['label'].map(lambda s: rr[s][k])
rc['lp']=rc['latency_p95_ms_recalculado'].map(math.log); rc['lt']=rc['t_conv_ms_mean_recalculado'].map(math.log)
g=rc.groupby(['carga','N','workers']).agg(n=('label','size'), lp=('lp','mean'), lt=('lt','mean'), thr=('throughput_rps_recalculado','mean'), req=('requests_total_recalculado','sum'), err=('errors_recalculado','sum')).reset_index()
g['p95_geo_r']=g['lp'].map(math.exp); g['tconv_geo_r']=g['lt'].map(math.exp)
e=pd.read_csv('http/oe4/esperado_celdas.csv')
m=g.merge(e,left_on=['carga','N','workers'],right_on=['carga','N','workers'])
d=lambda x,y: float((m[x]-m[y]).abs().max())
print('OE4 por celda (36 celdas, 6 bloques): celdas comparadas', len(m), '| dif abs máx | P95 geométrica:', d('p95_geo_r','p95_geo'), '| T_conv geométrica:', d('tconv_geo_r','tconv_geo'), '| throughput media:', d('thr','thr_media'), '| peticiones:', d('req','peticiones'), '| errores:', d('err','errores'))

## Etapa 2: análisis desde los indicadores por corrida

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('oe4/hashes_paquete_oe4.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe4',nm); n+=1
    if sha(p)!=h: bad.append(nm)
print('archivos del paquete verificados contra el hash local:', n, '| distintos:', bad); assert not bad
bad2=[]; ok2=[]
for l in open('oe4/campana_01/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe4/campana_01',nm)
    if os.path.exists(p):
        ok2.append(nm)
        if sha(p)!=h: bad2.append(nm)
print('contra el SHA256SUMS.txt de la campana, archivos presentes:', ok2, '| distintos:', bad2); assert not bad2

## Conteo de corridas

In [ ]:
import json, pandas as pd
R=[json.loads(l) for l in open('oe4/campana_01/runs.jsonl',encoding='utf8') if l.strip()]
r=pd.DataFrame(R); print('registros:', len(r), '| validos:', int((r.valid==True).sum()), '| no validos:', int((r.valid!=True).sum()))
print(r.groupby(['load','n_particles','workers']).block.nunique().value_counts().to_dict(), '(celdas por numero de bloques)')

## 1. Análisis PRE-REGISTRADO (`analisis_oe4.py`, mismos argumentos que en local: runs, salida, monitor y referencia OE2-B descriptiva)

In [ ]:
os.makedirs('oe4/salida_colab',exist_ok=True)
sh(f"python oe4/analisis_oe4.py oe4/campana_01/runs.jsonl oe4/salida_colab/preregistrado oe4/monitor.csv oe4/runs_oe2b_referencia.jsonl > oe4/salida_colab/log_preregistrado.txt 2>&1")
print(open('oe4/salida_colab/log_preregistrado.txt').read()[-1500:])

## 2. Análisis EXPLORATORIO de interferencia (NO pre-registrado)

In [ ]:
sh(f"python oe4/exploratorio_interferencia_oe4.py oe4/campana_01/runs.jsonl oe4/salida_colab/preregistrado/resultados_oe4.json oe4/salida_colab/exploratorio_interferencia_oe4.json > oe4/salida_colab/log_exploratorio.txt 2>&1")
print(open('oe4/salida_colab/log_exploratorio.txt').read()[-700:])

## 3. Comparación Colab vs local (diferencias absolutas y relativas)

In [ ]:
import json, math
import pandas as pd
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
def comparar_json(ruta_colab, ruta_local, etiqueta):
    A=dict(hoja(json.load(open(ruta_colab,encoding='utf8')))); B=dict(hoja(json.load(open(ruta_local,encoding='utf8'))))
    mismas = set(A)==set(B); mx=0.0; mxrel=0.0; peor=None; n_num=0; n_dist=0; n_nonum=0; dist=[]
    for k in A:
        if k not in B: continue
        va, vb = A[k], B[k]
        if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool) and not isinstance(vb,bool):
            n_num+=1; dif=abs(va-vb); rel=dif/max(abs(vb),1e-300) if dif>0 else 0.0
            if dif>mx: mx=dif; peor=(k,va,vb)
            mxrel=max(mxrel, rel)
            if dif>0: n_dist+=1
            if dif>1e-9 and rel>1e-9: dist.append((k,va,vb,dif,rel))
        else:
            n_nonum+=1
            if va!=vb: dist.append((k,va,vb,None,None))
    print(f'[{etiqueta}] mismas claves: {mismas} | hojas: {len(A)} (numericas {n_num}, no numericas {n_nonum})')
    print(f'   diferencia absoluta maxima: {mx:.3e} | relativa maxima: {mxrel:.3e} | hojas numericas con diferencia > 0: {n_dist} | hojas con diferencia > 1e-9 (abs y rel): {len(dist)}')
    print('   peor hoja:', peor)
    for d in dist[:20]: print('   DIFERENCIA', d)
    return dict(etiqueta=etiqueta, mismas_claves=mismas, hojas=len(A), numericas=n_num, dif_abs_max=mx, dif_rel_max=mxrel, hojas_con_dif=n_dist, hojas_dif_gt_1e9=len(dist))
def comparar_csv(ruta_colab, ruta_local, etiqueta):
    a=pd.read_csv(ruta_colab); b=pd.read_csv(ruta_local)
    ok = list(a.columns)==list(b.columns) and a.shape==b.shape
    mx=0.0; mxrel=0.0; nd=0
    if ok:
        for c in a.columns:
            if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c]) and not pd.api.types.is_bool_dtype(a[c]) and not pd.api.types.is_bool_dtype(b[c]):
                dif=(a[c]-b[c]).abs().fillna(0); mx=max(mx,float(dif.max()))
                rel=(dif/b[c].abs().clip(lower=1e-300)).where(dif>0,0.0); mxrel=max(mxrel,float(rel.max())); nd+=int((dif>0).sum())
            else:
                nd+=int((a[c].astype(object).where(a[c].notna(),'<NA>').astype(str)!=b[c].astype(object).where(b[c].notna(),'<NA>').astype(str)).sum())
    print(f'[{etiqueta}] misma forma y columnas: {ok} {a.shape} | dif abs max: {mx:.3e} | rel max: {mxrel:.3e} | celdas distintas: {nd}')
    return dict(etiqueta=etiqueta, misma_forma=ok, filas=a.shape[0], dif_abs_max=mx, dif_rel_max=mxrel, celdas_distintas=nd)

In [ ]:
res=[]
res.append(comparar_json('oe4/salida_colab/preregistrado/resultados_oe4.json','oe4/esperado_local/resultados_oe4.json','OE4 pre-registrado: resultados_oe4.json'))
for f in ('celdas','contrastes','anova_p95','anova_thr','anova_tconv'):
    res.append(comparar_csv(f'oe4/salida_colab/preregistrado/{f}.csv',f'oe4/esperado_local/{f}.csv',f'OE4 pre-registrado: {f}.csv'))
res.append(comparar_json('oe4/salida_colab/exploratorio_interferencia_oe4.json','oe4/esperado_local/exploratorio_interferencia_oe4.json','OE4 EXPLORATORIO: interferencia'))
pd.DataFrame(res).to_csv('oe4/salida_colab/comparacion_colab_vs_local.csv',index=False)
print(pd.DataFrame(res)[['etiqueta','dif_abs_max','dif_rel_max']].to_string())

In [ ]:
import hashlib
print('SHA-256 de los 6 archivos de resultados (Colab == local):')
for f in ('anova_p95.csv','anova_thr.csv','anova_tconv.csv','celdas.csv','contrastes.csv','resultados_oe4.json'):
    print(f, sha('oe4/salida_colab/preregistrado/'+f)==sha('oe4/esperado_local/'+f))

## 4. Indicadores principales (Colab | local)

In [ ]:
a=json.load(open('oe4/salida_colab/preregistrado/resultados_oe4.json',encoding='utf8')); b=json.load(open('oe4/esperado_local/resultados_oe4.json',encoding='utf8'))
print('registros/validas/no validas/celdas con 6 bloques/interferidas:', [a[k] for k in ('n_registros','n_validas','n_no_validas','celdas_con_6_bloques','interferidas')], '|', [b[k] for k in ('n_registros','n_validas','n_no_validas','celdas_con_6_bloques','interferidas')])
print('sensibilidad sin interferidas:', a['sensibilidad_sin_interferidas'])
print('contrastes significativos (Holm < 0.05) por respuesta|familia:'); print(pd.DataFrame({'colab':{k:v['significativos_holm_005'] for k,v in a['contrastes_resumen'].items()},'local':{k:v['significativos_holm_005'] for k,v in b['contrastes_resumen'].items()}}).to_string())
print('eta2 parcial por respuesta (terminos principales):')
for rk in ('p95','thr','tconv'):
    ta={t['termino']:t for t in a['anova'][rk]['terminos']}; tb={t['termino']:t for t in b['anova'][rk]['terminos']}
    print(rk, {t:(round(ta[t]['eta2_parcial'],6), round(tb[t]['eta2_parcial'],6)) for t in ('C(load)','C(N)','C(workers)','C(load):C(workers)','C(block)')})
c=pd.read_csv('oe4/salida_colab/preregistrado/celdas.csv'); cl=pd.read_csv('oe4/esperado_local/celdas.csv')
print('celdas P95<2s (<=25 usuarios), Colab vs local:'); print(pd.concat([c[c.carga<=25].groupby(['workers','N']).p95_lt_2s.sum().rename('colab'), cl[cl.carga<=25].groupby(['workers','N']).p95_lt_2s.sum().rename('local')],axis=1).to_string())
print('celdas con throughput IC inferior >= 20 req/s:', int(c.thr_ge_20.sum()), int(cl.thr_ge_20.sum()))
print('estabilidad (Colab):', json.dumps({k:v for k,v in a['estabilidad'].items() if k in ('p95','thr','tconv')})[:600])

## 4. Comparación con las cifras de la tesis
Cada cifra se lee del resultado **reproducido** y se compara con la que figura en la tesis (versión CIERRE_FINAL) y en el JSON oficial sellado (`manifests/valores_oficiales.json`).

In [ ]:
from repro import official, validation, tables, figures
from repro.data_loading import sha256_file
import pandas as pd
RUTAS = {'resultados_oe4.json': 'oe4/salida_colab/preregistrado/resultados_oe4.json', 'celdas.csv': 'oe4/salida_colab/preregistrado/celdas.csv', 'intentos.jsonl': 'oe4/campana_01/intentos.jsonl'}
cifras = official.evaluar('OE4', RUTAS)
print(cifras[['id', 'descripcion', 'esperado', 'obtenido', 'dif', 'estado']].to_string(index=False))


## 5. Clasificación de las diferencias frente al resultado local
Exacta · dentro de tolerancia (1e-9) · explicada por versión · no explicada · no reproducible.

In [ ]:
import scipy
REG = validation.Registro(scipy_version=scipy.__version__)
COMPARACIONES = [('OE4 resultados_oe4.json', 'oe4/salida_colab/preregistrado/resultados_oe4.json', 'oe4/esperado_local/resultados_oe4.json'), ('OE4 celdas.csv', 'oe4/salida_colab/preregistrado/celdas.csv', 'oe4/esperado_local/celdas.csv'), ('OE4 contrastes.csv', 'oe4/salida_colab/preregistrado/contrastes.csv', 'oe4/esperado_local/contrastes.csv'), ('OE4 anova_p95.csv', 'oe4/salida_colab/preregistrado/anova_p95.csv', 'oe4/esperado_local/anova_p95.csv'), ('OE4 anova_thr.csv', 'oe4/salida_colab/preregistrado/anova_thr.csv', 'oe4/esperado_local/anova_thr.csv'), ('OE4 anova_tconv.csv', 'oe4/salida_colab/preregistrado/anova_tconv.csv', 'oe4/esperado_local/anova_tconv.csv'), ('OE4 exploratorio_interferencia_oe4.json', 'oe4/salida_colab/exploratorio_interferencia_oe4.json', 'oe4/esperado_local/exploratorio_interferencia_oe4.json')]
for _e, _a, _b in COMPARACIONES:
    REG.comparar(_e, _a, _b)


## 6. Tablas

In [ ]:
OUT = 'outputs'
TABLAS = {'celdas': 'oe4/salida_colab/preregistrado/celdas.csv', 'contrastes': 'oe4/salida_colab/preregistrado/contrastes.csv', 'anova_p95': 'oe4/salida_colab/preregistrado/anova_p95.csv', 'anova_thr': 'oe4/salida_colab/preregistrado/anova_thr.csv', 'anova_tconv': 'oe4/salida_colab/preregistrado/anova_tconv.csv'}
tablas_generadas = tables.exportar_tablas('OE4', TABLAS, os.path.join(OUT, 'tables'))
print('Tablas generadas:')
for _t in tablas_generadas:
    print('  ', _t)


## 7. Gráficos
Se generan desde la salida del análisis ejecutado arriba (no desde valores copiados). Se exportan PNG (300 ppi), SVG y el CSV de la serie dibujada (`figure_data/`).

In [ ]:
METAS = official.cargar_valores()['metas_de_diseno']
fig = figures.fig_oe4(RUTAS['celdas.csv'], os.path.join(OUT, 'figures', 'OE4'), metas=METAS)
ref = figures.fig_oe4('oe4/esperado_local/celdas.csv', os.path.join(OUT, 'figures_ref_desde_resultado_local', 'OE4'), metas=METAS)
REG.comparar('Serie de la Figura 4 reproducida frente a la serie dibujada desde el resultado oficial local', fig['serie_csv'], ref['serie_csv'])
_t = official.cargar_valores()['figuras_tesis']['fig_oe4']
IDENTICA = sha256_file(fig['png']) == _t['sha256_png_tesis']
print('Figura 4: PNG identico bit a bit al de la tesis (pagina %s del indice): %s' % (_t['pagina_indice'], IDENTICA))
if not IDENTICA:
    print('  Esperable fuera del entorno de la tesis (matplotlib 3.10.3 y fuente Arial): la imagen puede diferir en pixeles; la comprobacion valida es la de la serie dibujada.')
try:
    from IPython.display import Image, display
    display(Image(fig['png']))
except Exception:
    pass


## 8. Pruebas automáticas de consistencia

In [ ]:
fallos = []
if cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE']).any():
    fallos.append('cifras oficiales: ' + str(cifras[cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE'])][['id', 'estado']].values.tolist()))
peor, cuenta = REG.veredicto()
if peor in ('NO_EXPLICADA', 'NO_REPRODUCIBLE'):
    fallos.append('comparacion con el resultado local: ' + peor)
if not (os.path.getsize(fig['png']) > 0 and os.path.getsize(fig['serie_csv']) > 0):
    fallos.append('figura vacia')
assert not fallos, fallos
print('PRUEBAS AUTOMATICAS DE CONSISTENCIA: todas superadas | cifras oficiales comprobadas:', len(cifras), '| comparaciones:', cuenta)


## 9. Resumen final de la reproducción

In [ ]:
import shutil
resumen = tables.guardar_resumen('OE4', os.path.join(OUT, 'reports'), REG.tabla(), cifras, ENTORNO, extra=dict(veredicto=peor, conteo=cuenta, paquetes=PAQUETES))
print('RESUMEN DE REPRODUCCION OE4')
print(REG.tabla()[['comparacion', 'categoria', 'dif_abs_max']].to_string(index=False))
print('Veredicto (peor categoria):', peor, cuenta)
print('Cifras de la tesis comprobadas: %d (%d coinciden exactamente, %d dentro del redondeo)' % (len(cifras), (cifras.estado == 'COINCIDE').sum(), (cifras.estado == 'COINCIDE_REDONDEO').sum()))
print('LIMITACIONES: 216 corridas validas de 220 intentos, todas con interferencia ambiental; analisis por bloques; la campana no es controlada.')
print('Alcance: se reproduce el analisis, no la adquisicion de datos; no se repite ninguna campana.')
_z = shutil.make_archive('resultados_OE4', 'zip', OUT)
print('Salidas empaquetadas en', os.path.basename(_z))
if IN_COLAB:
    from google.colab import files
    files.download(_z)
